In [ ]:
from all_functions import *
from ngc import *

In [ ]:
X_train = pd.read_csv('.../train.csv', sep=',', index_col=0)
list_dir = '.../list.txt'
var_list = get_var_list(list_dir)
X_arr = X_train[var_list].to_numpy(dtype=float)
best, results, S_avg= tune_ngc_random(
    normal_runs_X=[X_arr],
    n_trials=30,
    stability_weight=0.1,
    max_iter=500,
    check_every=200,
    standardize=False,
    subsample_T=None,
    seed=42,
    min_edges=1,
    max_edges=None
)
print("BEST:", best)

# Saving causal matrix

In [ ]:
hparams = {
    "lag": best["hp"]["lag"],
    "hidden": best["hp"]["hidden"],
    "lam": best["hp"]["lam"],
    "lam_r": best["hp"]["lam_r"],
    "lr": best["hp"]["lr"],
    "standardize": False,
    "penalty": "GL",
}


save_ngc_results(
    save_dir=".../NGC_learning/",
    S=S_avg,            
    feature_names=var_list,
    hparams=hparams
)

# Causal graph extraction

In [ ]:
S, _, _, _, _, _ = load_ngc_results(".../NGC_learning")
edges, edge_index = build_graph_row_quantile(
    S,
    row_q=0.95,
    remove_self=True,
    abs_threshold=None,
    ensure_one_edge=True
)

edge_index = adding_self_loop(S, edge_index)

# Save the causal graph

In [ ]:
base_dir = Path(r"C:/Users/uig53331/OneDrive - Aumovio SE/Bureau/THESE/codes/GDN_PROJECT/notebooks/SWaT Dataset/graphs")
save_graph(base_dir,
    edge_index=edge_index, 
    folder_name="",
    graph_name="",
    feature_map=var_list,
    extra_meta={"source": "NGC"}
)